In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("contract-lakehouse-environment-validation")
    .master("spark://spark-master:7077")
    .config("spark.jars", "/opt/spark/jars_extra/hadoop-aws-3.3.4.jar,/opt/spark/jars_extra/aws-java-sdk-bundle-1.12.262.jar")
    .config("spark.driver.extraClassPath", "/opt/spark/jars_extra/*")
    .config("spark.executor.extraClassPath", "/opt/spark/jars_extra/*")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "admin12345")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

spark.version

26/06/22 20:47:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


'3.5.1'

In [4]:
## Criação do bucket

import boto3

s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="admin12345",
    region_name="us-east-1",
)

bucket_name = "contracts"

existing_buckets = [
    bucket["Name"]
    for bucket in s3_client.list_buckets()["Buckets"]
]

if bucket_name not in existing_buckets:
    s3_client.create_bucket(Bucket=bucket_name)

print("Buckets:", [bucket["Name"] for bucket in s3_client.list_buckets()["Buckets"]])

Buckets: ['contracts']


## Próximo teste (mais crítico da Fase 1)

In [5]:
## Criar DataFrame

data = [
    ("5900055119", "Contrato A", 1000000.0),
    ("5900086165", "Contrato B", 2500000.0),
]

df = spark.createDataFrame(
    data,
    ["contract_number", "contract_name", "contract_value"]
)

df.show()

+---------------+-------------+--------------+
|contract_number|contract_name|contract_value|
+---------------+-------------+--------------+
|     5900055119|   Contrato A|     1000000.0|
|     5900086165|   Contrato B|     2500000.0|
+---------------+-------------+--------------+



In [6]:
df.show()

+---------------+-------------+--------------+
|contract_number|contract_name|contract_value|
+---------------+-------------+--------------+
|     5900055119|   Contrato A|     1000000.0|
|     5900086165|   Contrato B|     2500000.0|
+---------------+-------------+--------------+



In [7]:
import sys
import pyspark

print(sys.version)
print(pyspark.__version__)

3.8.20 (default, Sep 27 2024, 06:05:23) 
[GCC 12.2.0]
3.5.1


In [8]:
output_path = "s3a://contracts/test/environment_validation/"

df.write.mode("overwrite").parquet(output_path)

spark.read.parquet(output_path).show()

26/06/21 16:53:22 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+---------------+-------------+--------------+
|contract_number|contract_name|contract_value|
+---------------+-------------+--------------+
|     5900055119|   Contrato A|     1000000.0|
|     5900086165|   Contrato B|     2500000.0|
+---------------+-------------+--------------+

